In [5]:
import pandas as pd
from scipy.special import comb
from scipy.stats import ttest_ind
from statsmodels.stats.proportion import proportions_ztest
import ast
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
soccer_ternary = pd.read_csv('../data/processed/soccer.csv', index_col=0)
soccer_ternary

,League,Season,Team,Sequence
0,Bundesliga,2000,Bayern Munich,"[3, 3, 3, 0, 3, 3, 0, 0, 3, 1, 3, 0, 0, 1, 3, ..."
1,Bundesliga,2000,Bochum,"[3, 0, 0, 0, 3, 1, 3, 0, 0, 0, 1, 0, 1, 0, 3, ..."
2,Bundesliga,2000,Cottbus,"[0, 0, 0, 3, 0, 0, 1, 3, 0, 3, 1, 3, 0, 0, 3, ..."
3,Bundesliga,2000,Dortmund,"[3, 3, 0, 3, 3, 0, 1, 3, 0, 0, 0, 3, 1, 3, 3, ..."
4,Bundesliga,2000,Ein Frankfurt,"[3, 0, 3, 0, 3, 1, 1, 0, 0, 3, 0, 3, 3, 0, 0, ..."
...,...,...,...,...
1433,La_Liga,2024,Sociedad,"[0, 3, 0, 1, 0, 0, 1, 3, 1, 3, 0, 3, 3, 0, 3, ..."
1434,La_Liga,2024,Valencia,"[0, 0, 0, 1, 0, 3, 1, 0, 1, 0, 1, 3, 0, 0, 0, ..."
1435,La_Liga,2024,Valladolid,"[3, 0, 1, 0, 0, 1, 0, 0, 0, 3, 0, 0, 1, 0, 0, ..."
1436,La_Liga,2024,Vallecano,"[3, 1, 0, 0, 3, 1, 1, 1, 3, 0, 3, 0, 0, 0, 3, ..."


In [9]:
def prob(sequence):
    return sum(sequence) / len(sequence)

def split(sequence):

    n1 = math.ceil(len(sequence) / 2)

    s1 = sequence[:n1] # includes the first n1 elements
    s2 = sequence[n1:] # the rest of the elements

    return s1,s2

In [11]:
def c2(sequence):
    s1, s2 = split(sequence)

    k1 = sum(s1)
    k2 = sum(s2)

    n1 = len(s1)*3
    n2 = len(s2)*3

    _, p = proportions_ztest([k1, k2], [n1, n2], alternative = "larger")

    if (p == 0):
        return math.inf
    else:
        return 1 / p

In [14]:
soccer_ternary['Sequence_Numeric'] = soccer_ternary['Sequence'].apply(ast.literal_eval)

In [16]:
## Add Columns n, n1, n2, k, k1, k2

soccer_ternary['n'] = soccer_ternary['Sequence_Numeric'].apply(len)

splits = soccer_ternary['Sequence_Numeric'].apply(split)

soccer_ternary['n1'] = splits.apply(lambda x: len(x[0]))
soccer_ternary['n2'] = splits.apply(lambda x: len(x[1]))

soccer_ternary['k'] = soccer_ternary['Sequence_Numeric'].apply(sum)
soccer_ternary['k1'] = splits.apply(lambda x: sum(x[0]))
soccer_ternary['k2'] = splits.apply(lambda x: sum(x[1]))
# Apply C1

#soccer_ternary['c1'] = [c1(seq) for seq in soccer_ternary['Sequence_Numeric']]
# Apply C2

soccer_ternary['c2'] = [c2(seq) for seq in soccer_ternary['Sequence_Numeric']]

/opt/anaconda3/lib/python3.9/site-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std


In [18]:
soccer_ternary

,League,Season,Team,Sequence,Sequence_Numeric,n,n1,n2,k,k1,k2,c2
0,Bundesliga,2000,Bayern Munich,"[3, 3, 3, 0, 3, 3, 0, 0, 3, 1, 3, 0, 0, 1, 3, ...","[3, 3, 3, 0, 3, 3, 0, 0, 3, 1, 3, 0, 0, 1, 3, ...",34,17,17,63,30,33,1.370834
1,Bundesliga,2000,Bochum,"[3, 0, 0, 0, 3, 1, 3, 0, 0, 0, 1, 0, 1, 0, 3, ...","[3, 0, 0, 0, 3, 1, 3, 0, 0, 0, 1, 0, 1, 0, 3, ...",34,17,17,27,18,9,46.089683
2,Bundesliga,2000,Cottbus,"[0, 0, 0, 3, 0, 0, 1, 3, 0, 3, 1, 3, 0, 0, 3, ...","[0, 0, 0, 3, 0, 0, 1, 3, 0, 3, 1, 3, 0, 0, 3, ...",34,17,17,39,17,22,1.182258
3,Bundesliga,2000,Dortmund,"[3, 3, 0, 3, 3, 0, 1, 3, 0, 0, 0, 3, 1, 3, 3, ...","[3, 3, 0, 3, 3, 0, 1, 3, 0, 0, 0, 3, 1, 3, 3, ...",34,17,17,58,30,28,2.901612
4,Bundesliga,2000,Ein Frankfurt,"[3, 0, 3, 0, 3, 1, 1, 0, 0, 3, 0, 3, 3, 0, 0, ...","[3, 0, 3, 0, 3, 1, 1, 0, 0, 3, 0, 3, 3, 0, 0, ...",34,17,17,35,20,15,6.733019
...,...,...,...,...,...,...,...,...,...,...,...,...
1433,La_Liga,2024,Sociedad,"[0, 3, 0, 1, 0, 0, 1, 3, 1, 3, 0, 3, 3, 0, 3, ...","[0, 3, 0, 1, 0, 0, 1, 3, 1, 3, 0, 3, 3, 0, 3, ...",38,19,19,46,28,18,35.552617
1434,La_Liga,2024,Valencia,"[0, 0, 0, 1, 0, 3, 1, 0, 1, 0, 1, 3, 0, 0, 0, ...","[0, 0, 0, 1, 0, 3, 1, 0, 1, 0, 1, 3, 0, 0, 0, ...",38,19,19,46,13,33,1.000067
1435,La_Liga,2024,Valladolid,"[3, 0, 1, 0, 0, 1, 0, 0, 0, 3, 0, 0, 1, 0, 0, ...","[3, 0, 1, 0, 0, 1, 0, 0, 0, 3, 0, 0, 1, 0, 0, ...",38,19,19,16,15,1,12495.251146
1436,La_Liga,2024,Vallecano,"[3, 1, 0, 0, 3, 1, 1, 1, 3, 0, 3, 0, 0, 0, 3, ...","[3, 1, 0, 0, 3, 1, 1, 1, 3, 0, 3, 0, 0, 0, 3, ...",38,19,19,52,25,27,1.546616


In [20]:
soccer_ternary['Team_Name'] = soccer_ternary['Season'].astype(str) + ' ' + soccer_ternary['Team']
soccer_ternary

,League,Season,Team,Sequence,Sequence_Numeric,n,n1,n2,k,k1,k2,c2,Team_Name
0,Bundesliga,2000,Bayern Munich,"[3, 3, 3, 0, 3, 3, 0, 0, 3, 1, 3, 0, 0, 1, 3, ...","[3, 3, 3, 0, 3, 3, 0, 0, 3, 1, 3, 0, 0, 1, 3, ...",34,17,17,63,30,33,1.370834,2000 Bayern Munich
1,Bundesliga,2000,Bochum,"[3, 0, 0, 0, 3, 1, 3, 0, 0, 0, 1, 0, 1, 0, 3, ...","[3, 0, 0, 0, 3, 1, 3, 0, 0, 0, 1, 0, 1, 0, 3, ...",34,17,17,27,18,9,46.089683,2000 Bochum
2,Bundesliga,2000,Cottbus,"[0, 0, 0, 3, 0, 0, 1, 3, 0, 3, 1, 3, 0, 0, 3, ...","[0, 0, 0, 3, 0, 0, 1, 3, 0, 3, 1, 3, 0, 0, 3, ...",34,17,17,39,17,22,1.182258,2000 Cottbus
3,Bundesliga,2000,Dortmund,"[3, 3, 0, 3, 3, 0, 1, 3, 0, 0, 0, 3, 1, 3, 3, ...","[3, 3, 0, 3, 3, 0, 1, 3, 0, 0, 0, 3, 1, 3, 3, ...",34,17,17,58,30,28,2.901612,2000 Dortmund
4,Bundesliga,2000,Ein Frankfurt,"[3, 0, 3, 0, 3, 1, 1, 0, 0, 3, 0, 3, 3, 0, 0, ...","[3, 0, 3, 0, 3, 1, 1, 0, 0, 3, 0, 3, 3, 0, 0, ...",34,17,17,35,20,15,6.733019,2000 Ein Frankfurt
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1433,La_Liga,2024,Sociedad,"[0, 3, 0, 1, 0, 0, 1, 3, 1, 3, 0, 3, 3, 0, 3, ...","[0, 3, 0, 1, 0, 0, 1, 3, 1, 3, 0, 3, 3, 0, 3, ...",38,19,19,46,28,18,35.552617,2024 Sociedad
1434,La_Liga,2024,Valencia,"[0, 0, 0, 1, 0, 3, 1, 0, 1, 0, 1, 3, 0, 0, 0, ...","[0, 0, 0, 1, 0, 3, 1, 0, 1, 0, 1, 3, 0, 0, 0, ...",38,19,19,46,13,33,1.000067,2024 Valencia
1435,La_Liga,2024,Valladolid,"[3, 0, 1, 0, 0, 1, 0, 0, 0, 3, 0, 0, 1, 0, 0, ...","[3, 0, 1, 0, 0, 1, 0, 0, 0, 3, 0, 0, 1, 0, 0, ...",38,19,19,16,15,1,12495.251146,2024 Valladolid
1436,La_Liga,2024,Vallecano,"[3, 1, 0, 0, 3, 1, 1, 1, 3, 0, 3, 0, 0, 0, 3, ...","[3, 1, 0, 0, 3, 1, 1, 1, 3, 0, 3, 0, 0, 0, 3, ...",38,19,19,52,25,27,1.546616,2024 Vallecano


In [27]:
#c3
def c3(sequence, N):

    sequence = np.asarray(sequence)   # convert once
    M = 0

    _, s2 = split(sequence)
    k2 = np.sum(s2)

    for _ in range(N):
        shuffled = np.random.permutation(sequence)  # new shuffled copy
        _, tmp = split(shuffled)

        if np.sum(tmp) <= k2:
            M += 1

    return N / M if M != 0 else math.inf

In [45]:
N = 1000
soccer_ternary['c3'] = [c3(seq, N) for seq in soccer_ternary['Sequence_Numeric']]

In [47]:
## Recalculate top 100 in c3 column 

soccer_indexes = list(soccer_ternary.sort_values('c3', ascending = False).head(100).index)

N = 100000


mask = soccer_ternary.index.isin(soccer_indexes)
soccer_ternary.loc[mask, 'c3'] = soccer_ternary.loc[mask, 'Sequence_Numeric'].apply(lambda seq: c3(seq, N))

In [49]:
soccer_ternary

,League,Season,Team,Sequence,Sequence_Numeric,n,n1,n2,k,k1,k2,c2,Team_Name,log_c2,c3,log_c3
0,Bundesliga,2000,Bayern Munich,"[3, 3, 3, 0, 3, 3, 0, 0, 3, 1, 3, 0, 0, 1, 3, ...","[3, 3, 3, 0, 3, 3, 0, 0, 3, 1, 3, 0, 0, 1, 3, ...",34,17,17,63,30,33,1.370834,2000 Bayern Munich,0.315419,1.418440,0.356675
1,Bundesliga,2000,Bochum,"[3, 0, 0, 0, 3, 1, 3, 0, 0, 0, 1, 0, 1, 0, 3, ...","[3, 0, 0, 0, 3, 1, 3, 0, 0, 0, 1, 0, 1, 0, 3, ...",34,17,17,27,18,9,46.089683,2000 Bochum,3.830589,6.896552,1.924149
2,Bundesliga,2000,Cottbus,"[0, 0, 0, 3, 0, 0, 1, 3, 0, 3, 1, 3, 0, 0, 3, ...","[0, 0, 0, 3, 0, 0, 1, 3, 0, 3, 1, 3, 0, 0, 3, ...",34,17,17,39,17,22,1.182258,2000 Cottbus,0.167426,1.342282,0.286350
3,Bundesliga,2000,Dortmund,"[3, 3, 0, 3, 3, 0, 1, 3, 0, 0, 0, 3, 1, 3, 3, ...","[3, 3, 0, 3, 3, 0, 1, 3, 0, 0, 0, 3, 1, 3, 3, ...",34,17,17,58,30,28,2.901612,2000 Dortmund,1.065267,2.314815,0.820981
4,Bundesliga,2000,Ein Frankfurt,"[3, 0, 3, 0, 3, 1, 1, 0, 0, 3, 0, 3, 3, 0, 0, ...","[3, 0, 3, 0, 3, 1, 1, 0, 0, 3, 0, 3, 3, 0, 0, ...",34,17,17,35,20,15,6.733019,2000 Ein Frankfurt,1.907024,3.333333,1.203973
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1433,La_Liga,2024,Sociedad,"[0, 3, 0, 1, 0, 0, 1, 3, 1, 3, 0, 3, 3, 0, 3, ...","[0, 3, 0, 1, 0, 0, 1, 3, 1, 3, 0, 3, 3, 0, 3, ...",38,19,19,46,28,18,35.552617,2024 Sociedad,3.571014,6.896552,1.958995
1434,La_Liga,2024,Valencia,"[0, 0, 0, 1, 0, 3, 1, 0, 1, 0, 1, 3, 0, 0, 0, ...","[0, 0, 0, 1, 0, 3, 1, 0, 1, 0, 1, 3, 0, 0, 0, ...",38,19,19,46,13,33,1.000067,2024 Valencia,0.000067,1.001001,0.006018
1435,La_Liga,2024,Valladolid,"[3, 0, 1, 0, 0, 1, 0, 0, 0, 3, 0, 0, 1, 0, 0, ...","[3, 0, 1, 0, 0, 1, 0, 0, 0, 3, 0, 0, 1, 0, 0, ...",38,19,19,16,15,1,12495.251146,2024 Valladolid,9.433104,90.661831,5.115996
1436,La_Liga,2024,Vallecano,"[3, 1, 0, 0, 3, 1, 1, 1, 3, 0, 3, 0, 0, 0, 3, ...","[3, 1, 0, 0, 3, 1, 1, 1, 3, 0, 3, 0, 0, 0, 3, ...",38,19,19,52,25,27,1.546616,2024 Vallecano,0.436069,1.589825,0.397497


In [57]:
soccer_ternary_output = "../output/csvs/soccer_ternary_output.csv"

In [63]:
soccer_ternary_out = soccer_ternary[['League', 'Season', 'Team', 'Sequence', 'n', 'n1', 'n2', 'k', 'k1', 'k2', 'Team_Name', 'c2', 'c3']]

In [65]:
soccer_ternary_out.to_csv(soccer_ternary_output)